# 📈 Day 01a: Linear Regression — Math + Code From Scratch

---

## 🎯 What You'll Master Today
- Cost function (MSE) and its gradient
- Gradient descent derivation and implementation
- Normal equation (closed-form solution)
- Linear regression from scratch vs sklearn

**Goal:** Code gradient descent from memory in an interview.

---

```
╔═══════════════════════════════════════════════════════════════════╗
║  LINEAR REGRESSION — THE MATH                                    ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  Model:     ŷ = Xw + b     (or ŷ = Xw with bias in X)         ║
║                                                                   ║
║  Cost:      J(w) = (1/2n) Σ(ŷᵢ - yᵢ)²     (MSE / 2)          ║
║                                                                   ║
║  Gradient:  ∂J/∂w = (1/n) Xᵀ(Xw - y)                          ║
║                                                                   ║
║  Update:    w = w - α * ∂J/∂w                                   ║
║                                                                   ║
║  Normal Eq: w = (XᵀX)⁻¹ Xᵀy   (closed-form, no iteration)    ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
import numpy as np

# ============================================================
# Generate synthetic data: y = 3x₁ + 5x₂ + 7 + noise
# ============================================================

np.random.seed(42)
n_samples, n_features = 200, 2

X = np.random.randn(n_samples, n_features)
true_w = np.array([3.0, 5.0])
true_b = 7.0
y = X @ true_w + true_b + np.random.randn(n_samples) * 0.5

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"True weights: {true_w}, bias: {true_b}")

In [ ]:
# ============================================================
# ⭐ Linear Regression — From Scratch (Gradient Descent)
# ============================================================

class LinearRegressionScratch:
    def __init__(self, lr=0.01, n_iters=1000):
        self.lr = lr
        self.n_iters = n_iters
        self.weights = None
        self.bias = None
        self.losses = []
    
    def fit(self, X, y):
        n_samples, n_features = X.shape
        
        # Initialize weights
        self.weights = np.zeros(n_features)
        self.bias = 0
        
        for i in range(self.n_iters):
            # Forward: predictions
            y_pred = X @ self.weights + self.bias
            
            # Loss: MSE
            loss = np.mean((y_pred - y) ** 2)
            self.losses.append(loss)
            
            # Gradients
            dw = (2 / n_samples) * (X.T @ (y_pred - y))
            db = (2 / n_samples) * np.sum(y_pred - y)
            
            # Update
            self.weights -= self.lr * dw
            self.bias -= self.lr * db
            
            if i % 200 == 0:
                print(f"Iter {i:4d}: loss={loss:.4f}, w={self.weights.round(3)}, b={self.bias:.3f}")
        
        return self
    
    def predict(self, X):
        return X @ self.weights + self.bias

# Train
model = LinearRegressionScratch(lr=0.05, n_iters=1000)
model.fit(X, y)

print(f"\nLearned: w={model.weights.round(3)}, b={model.bias:.3f}")
print(f"True:    w={true_w}, b={true_b}")

In [ ]:
# ============================================================
# Normal Equation — Closed-Form Solution
# ============================================================

# w = (XᵀX)⁻¹ Xᵀy
# Add bias column to X
X_b = np.column_stack([np.ones(len(X)), X])  # [1, x1, x2]

# Normal equation
w_normal = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ y
# Better: use solve instead of inv
w_normal2 = np.linalg.solve(X_b.T @ X_b, X_b.T @ y)

print(f"Normal eq: bias={w_normal[0]:.3f}, w={w_normal[1:].round(3)}")
print(f"Solve:     bias={w_normal2[0]:.3f}, w={w_normal2[1:].round(3)}")
print(f"GD:        bias={model.bias:.3f}, w={model.weights.round(3)}")

print("\n💡 Normal equation: exact but O(n³) for features. GD: iterative but scales better.")
print("   Use normal eq when features < ~10k. GD for larger.")

In [ ]:
# ============================================================
# sklearn Comparison
# ============================================================

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

sk_model = LinearRegression()
sk_model.fit(X, y)

print(f"sklearn:  w={sk_model.coef_.round(3)}, b={sk_model.intercept_:.3f}")
print(f"Scratch:  w={model.weights.round(3)}, b={model.bias:.3f}")
print(f"True:     w={true_w}, b={true_b}")

# Metrics
y_pred_sk = sk_model.predict(X)
y_pred_scratch = model.predict(X)

print(f"\nsklearn MSE: {mean_squared_error(y, y_pred_sk):.4f}")
print(f"Scratch MSE: {mean_squared_error(y, y_pred_scratch):.4f}")
print(f"R² score:    {r2_score(y, y_pred_sk):.4f}")

---

## 🗺️ Linear Regression Pattern Map

```
╔═══════════════════════════════════════════════════════════════════╗
║  LINEAR REGRESSION — From Memory                                 ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  GRADIENT DESCENT RECIPE:                                        ║
║    1. Init weights = 0                                           ║
║    2. Forward: ŷ = X @ w + b                                    ║
║    3. Loss: MSE = mean((ŷ - y)²)                                ║
║    4. Gradients: dw = (2/n) * Xᵀ(ŷ - y), db = (2/n) * Σ(ŷ-y)║
║    5. Update: w -= lr * dw, b -= lr * db                        ║
║    6. Repeat until convergence                                   ║
║                                                                   ║
║  NORMAL EQUATION: w = (XᵀX)⁻¹ Xᵀy  (use solve, not inv)      ║
║                                                                   ║
║  WHEN TO USE WHAT:                                               ║
║    Features < 10k → Normal equation                             ║
║    Features > 10k → Gradient descent                            ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | Derive gradient for MSE? | J = (1/2n)Σ(Xw-y)², ∂J/∂w = (1/n)Xᵀ(Xw-y) |
| 2 | GD vs Normal equation? | GD: iterative, O(knd), scales to big data. Normal: exact, O(n³ features), fails with singular XᵀX |
| 3 | What if features >> samples? | Use regularization (Ridge/Lasso). Normal eq may be singular |
| 4 | How to choose learning rate? | Too high: diverge. Too low: slow. Try 0.001, 0.01, 0.1. Use learning rate scheduling |
| 5 | Assumptions of linear regression? | Linearity, independence, homoscedasticity, normality of residuals |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Forward → Loss → Gradient → Update                  │
## │  • dw = (2/n) * Xᵀ(Xw - y)                            │
## │  • Normal equation for small features                   │
## │  • Use np.linalg.solve over inv                         │
## │  • Memorize the 6-step GD recipe                        │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Logistic Regression** — sigmoid, cross-entropy, binary classifier from scratch